In [7]:
import pandas as pd

In [8]:
general_df = pd.read_csv('./generalizability/ldaMataveMetrics.csv')

In [9]:
calibrated_dfs = []
for dataset in general_df['dataset'].value_counts().keys().tolist():
    temp_df = general_df[general_df['dataset'] == dataset]
    temp_df = temp_df.drop(columns='dataset')

    # Get metric columns. 
    metric_cols = [c for c in temp_df.columns if 'model' not in c]
    # Extract real rows.
    real_mask = temp_df['model'].str.startswith('real')
    real_df = temp_df.loc[real_mask, metric_cols]

    # Get mean and standard deviation. 
    mean = real_df.mean()
    std_dev = real_df.std(ddof=1)
    # Avoid divide by zero.
    std_dev = std_dev.replace(0, 1e-8)



    # Reliability is inverse variance. 
    reliability = 1 / std_dev**2

    weights = reliability / reliability.sum()
    weight_rank = weights.rank(ascending=False, method="dense").astype(int)
    # Calibrate scores. 
    calibrated = temp_df.copy()
    for m in metric_cols:
        calibrated[m] = ((temp_df[m] - mean[m]) / std_dev[m])
        calibrated[f"rank_{m}"] = weight_rank[m]

    calibrated["final_score"] = (
        calibrated[metric_cols] * weights
    ).sum(axis=1)

    calibrated['dataset'] = dataset

    calibrated_dfs.append(calibrated)

In [10]:
calibrated_results = pd.concat(calibrated_dfs)
calibrated_results

,model,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,...,rank_CLASSIFIER,rank_IRPR,rank_FID,rank_PR,rank_DC,rank_MAUVE,rank_TRADITIONAL,rank_ZERO,final_score,dataset
0,LDA,-5.636972e+06,45.287521,3.671598,11.564781,18.570181,204.958881,9.079844,117.144949,14.180594,...,10,3,7,4,8,6,9,2,-5.636972e+06,yahoo
1,MATAVE,-8.340858e+05,84.609292,4.307750,15.002829,20.686738,220.714097,8.272847,127.382847,26.562514,...,10,3,7,4,8,6,9,2,-8.340858e+05,yahoo
2,combinedTopicModel,-4.628214e+04,59.422155,3.548525,13.679438,15.818661,123.243023,9.194088,118.322146,16.165381,...,10,3,7,4,8,6,9,2,-4.628214e+04,yahoo
3,realToReal,0.000000e+00,-1.052020,-0.382261,1.132477,1.141413,-0.006836,1.151035,-0.622905,-0.786369,...,10,3,7,4,8,6,9,2,-3.438382e-11,yahoo
4,realToReal2,0.000000e+00,0.113771,1.134744,-0.370991,-0.721977,-0.996565,-0.655138,1.153469,-0.339085,...,10,3,7,4,8,6,9,2,-3.584212e-11,yahoo
5,realToReal3,0.000000e+00,0.938249,-0.752484,-0.761486,-0.419436,1.003400,-0.495896,-0.530564,1.125454,...,10,3,7,4,8,6,9,2,7.022594e-11,yahoo
6,LDA,-3.880162e+03,43.444244,3.207581,15.487224,56.088133,31.321924,88.215020,128.074400,32.591137,...,10,4,6,9,5,7,8,2,-3.731196e+03,banking77
7,MATAVE,-3.856662e+03,60.190278,3.876452,23.293524,78.692933,39.131934,185.066338,162.015016,45.129464,...,10,4,6,9,5,7,8,2,-3.707385e+03,banking77
8,combinedTopicModel,-1.953765e+02,56.346192,3.440207,18.923769,57.913300,27.536213,127.498001,119.862121,37.442921,...,10,4,6,9,5,7,8,2,-1.856855e+02,banking77
9,realToReal,5.773503e-01,1.093215,0.968353,-0.076028,0.231627,0.145823,-0.417914,-0.556193,-0.404111,...,10,4,6,9,5,7,8,2,5.724558e-01,banking77


In [11]:
calibrated_results.to_csv('./generalizability.csv')

In [14]:
ranked_dict_results = []
for dataset in calibrated_results['dataset'].unique():
    subset = calibrated_results[calibrated_results['dataset'] == dataset]
    # Extract not real rows.
    real_mask = subset['model'].str.startswith('real')
    subset = subset.loc[~real_mask]
    ranks = subset['final_score'].rank(
        ascending=False,
        method='dense'
    ).astype(int)
    ranked_dict_results.append(pd.DataFrame({
        'model': subset['model'].values,
        'dataset': dataset,
        'final_score': subset['final_score'].values,
        'rank': ranks.values
    }))

In [15]:
pd.concat(ranked_dict_results)

,model,dataset,final_score,rank
0,LDA,yahoo,-5.636972e+06,3
1,MATAVE,yahoo,-8.340858e+05,2
2,combinedTopicModel,yahoo,-4.628214e+04,1
0,LDA,banking77,-3.731196e+03,3
1,MATAVE,banking77,-3.707385e+03,2
2,combinedTopicModel,banking77,-1.856855e+02,1
0,LDA,medicalAbstracts,-1.000000e+08,1
1,MATAVE,medicalAbstracts,-1.000000e+08,1
2,combinedTopicModel,medicalAbstracts,-1.000000e+08,1
0,LDA,dementiaAudio,-1.000000e+08,3
